In [9]:
# grid search ARIMA parameters for time series
import warnings
import traceback
import pandas as pd
from math import sqrt
from pandas import read_csv
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, root_mean_squared_error


# evaluate an ARIMA model for a given order (p,d,q)
def evaluate_arima_model(df, arima_order):
    # prepare training dataset
    train_size = int(len(df) * 0.8)
    train, test = df[:train_size], df[train_size:]
    history = [x for x in train]
    # make predictions
    predictions = list()
    for t in range(len(test)):
        model = ARIMA(history, order=arima_order)
        model_fit = model.fit()
        yhat = model_fit.forecast()[0]
        predictions.append(yhat)
        history.append(test.iloc[t])
    # calculate out of sample error
    # rmse = sqrt(mean_squared_error(test, predictions))
    mape = mean_absolute_percentage_error(test, predictions)
    return mape


# evaluate combinations of p, d and q values for an ARIMA model
def evaluate_models(dataset, p_values, d_values, q_values):
    dataset = dataset.astype("float32")
    best_score, best_cfg = float("inf"), None
    for p in p_values:
        for d in d_values:
            for q in q_values:
                order = (p, d, q)
                try:
                    mape = evaluate_arima_model(dataset, order)
                    if mape < best_score:
                        best_score, best_cfg = mape, order
                    print("ARIMA%s MAPE=%.3f" % (order, mape))
                except Exception as e:
                    print(f"ARIMA{order} failed: {e}")
                    traceback.print_exc()
                    continue
    print("Best ARIMA%s MAPE=%" % (best_cfg, best_score))


# load dataset
df = pd.read_csv("data/lip_0_x.csv")
print(df.head())
# evaluate parameters
p_values = [0, 1, 5, 10, 20]
d_values = range(0, 2)
q_values = range(0, 2)
warnings.filterwarnings("ignore")
evaluate_models(df["0_x"], p_values, d_values, q_values)

   time  0_x
0  0.00  637
1  0.04  638
2  0.08  638
3  0.12  638
4  0.16  638
ARIMA(0, 0, 0) MAPE=0.018
ARIMA(0, 0, 1) MAPE=0.009
ARIMA(0, 1, 0) MAPE=0.002
ARIMA(0, 1, 1) MAPE=0.002
ARIMA(1, 0, 0) MAPE=0.002
ARIMA(1, 0, 1) MAPE=0.002
ARIMA(1, 1, 0) MAPE=0.001
ARIMA(1, 1, 1) MAPE=0.001
ARIMA(5, 0, 0) MAPE=0.001
ARIMA(5, 0, 1) MAPE=0.001
ARIMA(5, 1, 0) MAPE=0.001
ARIMA(5, 1, 1) MAPE=0.001


KeyboardInterrupt: 